In [1]:
from rdflib import Graph, RDF
import requests
import os

def load_graph_from_source(source):
    """
    Loads a Turtle file from either a local path or a remote URL.
    Returns an rdflib.Graph().
    """

    g = Graph()

    # Case 1 — URL
    if source.startswith("http://") or source.startswith("https://"):
        print(f"Fetching DCAT 1.1 file from URL:\n  {source}\n")
        r = requests.get(source)
        r.raise_for_status()
        data = r.text
        g.parse(data=data, format="turtle")

    # Case 2 — Local file
    else:
        if not os.path.exists(source):
            raise FileNotFoundError(f"Local Turtle file not found: {source}")

        print(f"Loading DCAT 1.1 file from disk:\n  {source}\n")
        g.parse(source, format="turtle")

    return g


def step_through_dcat_classes(source):
    """
    Reads a DCAT-US 1.1 Turtle file (local or URL) and allows
    stepping through all classes present in the graph.
    """

    g = load_graph_from_source(source)
    print(f"Loaded {len(g)} RDF triples.\n")

    # ---------------------------------------
    # Collect classes found in file
    # ---------------------------------------
    classes = set(o for _, _, o in g.triples((None, RDF.type, None)))

    classes = sorted(classes, key=lambda x: str(x))

    print("Classes found in the DCAT file:")
    for c in classes:
        print("  -", c)

    print("\n----------------------------------------")
    print("Stepping through classes interactively...")
    print("----------------------------------------")

    # ---------------------------------------
    # Step through each class
    # ---------------------------------------
    for idx, cls in enumerate(classes, start=1):

        input(f"\n[{idx}/{len(classes)}] Press ENTER to inspect class:\n  {cls}")

        # Find instances of this class
        instances = list(g.subjects(RDF.type, cls))

        print(f"\nClass: {cls}")
        print(f"Instances found: {len(instances)}")

        # Show each instance + its properties
        for inst in instances:
            print(f"\n  Instance: {inst}")
            for p, o in g.predicate_objects(inst):
                print(f"    {p} → {o}")

        print("\n----------------------------------------")

    print("\nDone! All classes inspected.\n")


if __name__ == "__main__":
    # Example usage (edit this):
    # source = "catalog_1_1.ttl"
    # source = "https://raw.githubusercontent.com/.../catalog.ttl"

    source = input("\nEnter DCAT 1.1 file path or URL: ").strip()
    step_through_dcat_classes(source)


FileNotFoundError: Local Turtle file not found: 